### `ode_library.ipynb`  
*Created*: Sept 18, 2026 <br/>
*About*: A library of ODE functions used for testing and benchmarking solvers.

In [40]:
using OrdinaryDiffEq, CairoMakie, NBInclude, UnPack, Printf, Test, LinearAlgebra, LaTeXStrings, Statistics
import OrdinaryDiffEqCore: OrdinaryDiffEqAlgorithm  

In [32]:
@nbinclude("ode_solvers/explicit_runge_kutta/euler.ipynb")

euler (generic function with 2 methods)

In [14]:
struct ODETestProblem{F, U, S, P}
    f::F
    u0::U
    tspan::Tuple{Float64, Float64}
    exact_solution::S
    p::P
end 

function ODETestProblem(f::F, u0::U, tspan::NTuple{2,<:Real}; exact_solution::S = nothing, p::P = nothing) where {F,U,S,P}
    tspan = Float64.(tspan)
    return ODETestProblem(f, u0, tspan, exact_solution, p)
end

ODETestProblem

In [7]:
#Identity function
identity_rhs(u,p,t) = 1.0             #RHS  
identity_exact(t,p) = t               #Exact solution 
identity = ODETestProblem(identity_rhs, 1.0, (0.0, 10.0); exact_solution = identity_exact)

ODETestProblem{typeof(identity_rhs), Float64, typeof(identity_exact), Nothing}(Main.identity_rhs, 1.0, (0.0, 10.0), Main.identity_exact, nothing)

In [17]:
#Exponential function
exponential_rhs(u,p,t) = u          #RHS
exponential_exact(t,p) = exp(t)     #Exact solution
exponential = ODETestProblem(exponential_rhs, 1.0, (0.0, 10.0); exact_solution = exponential_exact);

In [9]:
#Sine function
sinusoid_rhs(u,p,t) = cos(t)      #RHS
sinusoid_exact(t,p) = sin(t)      #Exact solution
sinusoid = ODETestProblem(sinusoid_rhs, 0.0, (0.0, 10.0); exact_solution = sinusoid_exact);

In [10]:
#Damped oscillator
function damped_oscillator_rhs(u,p,t)
    y, z = u 
    dy = z 
    dz = -y - z/8
    
    du = [dy, dz]
    return du
end 

function damped_oscillator_exact(t,p)
    A = 2.0
    B = 2/sqrt(255)
    ω = sqrt(255) / 16
    return exp(-t/16) * (A*cos(ω*t) + B*sin(ω*t))
end 

damped_oscillator = ODETestProblem(damped_oscillator_rhs, 0.0, (0.0, 10.0); exact_solution = damped_oscillator_exact);

In [11]:
function lotka_volterra_rhs(u,p,t)
    @unpack α, β, δ, γ = p
    x, y = u

    dx = α*x - β*x*y
    dy = δ*x*y - γ*y
    
    du = [dx, dy]
    return du
end

lotka_volterra = ODETestProblem(lotka_volterra_rhs, 0.0, (0.0, 10.0));

In [12]:
function lorenz_rhs(u,p,t)
    @unpack σ, ρ, β = p
    x, y, z = u 
    du = [σ*(y - x), x*(ρ - z) - y, x*y - β*z]
    return du 
end

lorenz_system = ODETestProblem(lorenz_rhs, 0.0, (0.0, 10.0))

ODETestProblem{typeof(lorenz_rhs), Float64, Nothing, Nothing}(Main.lorenz_rhs, 0.0, (0.0, 10.0), nothing, nothing)

In [44]:
#Write function that 
function compare_solutions(prob::ODETestProblem, reference_alg::OrdinaryDiffEqAlgorithm, custom_alg::A; dt::Real = 0.01) where {A}

    """
    reference_alg :: algorithm from the OrdinaryDiffEq package 
    custom_alg :: algorithm that I wrote, that I'm testing 
    error_norm :: function for computing the norm of the difference of the two solutions 
    """
  
    @unpack f, u0, tspan, exact_solution, p = prob

    #STEP 1: Solve the ODE using the custom algorithm 
    custom_sol = custom_alg(f, u0, tspan; p = p, dt = dt)

    #STEP 2: Solve the ODE using the reference algorithm (an algorithm from OrdinaryDiffEq.jl)
    reference_sol = solve(ODEProblem(f, u0, tspan, p), reference_alg; adaptive = false, dt = dt)
       
    u_custom = custom_sol.u
    u_reference = reference_sol.u
    u_exact = exact_solution.(custom_sol.t, Ref(p))
    
    #STEP 3: Compute the L2 errors
    mean_l2_error_reference = mean(norm.(u_reference .- u_custom))
    mean_l2_error_exact = mean(norm.(u_exact .- u_custom))
    
    @printf("Mean L2 Error (reference soln) =  %.4e \n", mean_l2_error_reference)
    @printf("Mean L2 Error (exact soln) =  %.4e \n", mean_l2_error_exact)
    
    return (reference_sol = reference_sol, custom_sol = custom_sol)
     
end 

compare_solutions (generic function with 1 method)

In [45]:
prob = exponential
@unpack f, u0, tspan, exact_solution, p = prob

ODETestProblem{typeof(exponential_rhs), Float64, typeof(exponential_exact), Nothing}(Main.exponential_rhs, 1.0, (0.0, 10.0), Main.exponential_exact, nothing)

In [46]:
prob = exponential
reference_alg = Euler()   #from OrdinaryDiffEq.jl 
custom_alg = euler        #my implementation

@unpack reference_sol, custom_sol = compare_solutions(prob, reference_alg, custom_alg; dt = 0.01);

Mean L2 Error (reference soln) =  9.9782e-13 
Mean L2 Error (exact soln) =  9.6707e+01 


In [ ]:
function f(u,p,t)
    return u
end

u0 = 1.0
tspan = (0.0, 3.0)

sol = euler(f, u0, tspan; dt = 0.1)

In [ ]:
fig = Figure(size = (400,400))
ax = Axis(fig[1,1], xlabel = L"x", y